# Working with `.bendl` bundles

This tutorial is a companion to `using_ben_py.ipynb`. That notebook covers the
plain BEN/XBEN *streams* (`binary_ensemble.stream` + `binary_ensemble.codec`);
this one covers the **`.bendl` bundle** — the recommended, self-describing
container format — and walks the full `binary_ensemble.bundle` /
`binary_ensemble.graph` API, driving it with a live GerryChain ReCom run.

It is written in the `# %%` "percent" cell format, so you can step through it
cell-by-cell in VS Code / Jupyter (via Jupytext) or just run it top-to-bottom
as a plain script: `python using_bendl.py`.

## What is a bundle, and why use one?

A plain `.ben` file is *just* the assignment stream: a sequence of districting
plans, with no record of the graph they were drawn on. To use it, a
collaborator has to separately track down the matching dual-graph JSON **and**
know the exact node ordering the assignments were written in. Lose either and
the file is undecodable.

A `.bendl` bundle fixes this by wrapping the stream together with *assets* in a
single file:

- the **dual graph** (`graph.json`), so the file is self-describing;
- an optional **`node_permutation_map.json`**, recording any reordering applied
  to the graph for better compression;
- **`metadata.json`**, for run provenance (seed, parameters, generator, …);
- arbitrary **custom assets** (notes, analysis results, plots-as-bytes, …).

Intended use cases:

1. **Shareable, reproducible ensembles** — hand someone one file; they can
   recover the graph and replay the plans with no side files.
2. **Provenance** — stamp the seed / chain parameters into the bundle.
3. **Better compression** — reorder the graph (RCM / multi-level clustering)
   before writing so the BEN/XBEN delta-encoding shrinks; the permutation map
   keeps the reordering reversible.
4. **A lifecycle** — work in BEN (fast) while a project is active, then
   recompress the bundle to XBEN for long-term archival, assets preserved.
5. **Extensibility** — append analysis results to a finished bundle later,
   without rewriting the stream.

## Setup

We need a dual graph to draw plans on. Rather than download a multi-megabyte
real-world graph, we *generate* a `SIDE × SIDE` grid (here 32×32 = 1024 nodes) —
big enough to feel like a real ensemble, small enough to run in seconds, and
fully reproducible. Each node gets unit population (`TOTPOP = 1`) and an initial
`district` label of vertical stripes, which gives ReCom a contiguous, balanced
starting partition.

Then we deliberately **shuffle the node order**. Real-world dual graphs rarely
arrive in a compression-friendly order (think census blocks listed by GEOID, or
nodes in arbitrary shapefile order), so the stored order has no relationship to
graph locality. Shuffling reproduces that — and it's exactly the situation where
reordering before encoding pays off, which we'll see below. We write the graph
out as NetworkX adjacency JSON under `example_data/`, the shape a bundle stores.

In [1]:
import json
import random

import networkx as nx
from pathlib import Path

Path("example_data").mkdir(exist_ok=True)

SIDE, N_DISTRICTS = 32, 4  # 1024 nodes; SIDE must be divisible by N_DISTRICTS
GRAPH_PATH = Path("example_data/grid.json")


def build_grid_graph(side, n_districts, shuffle_seed=0):
    """A side*side grid with unit population, stripe districts, and a shuffled order."""
    g = nx.grid_2d_graph(side, side)
    g = nx.convert_node_labels_to_integers(g, ordering="sorted")  # row-major ints
    cols_per_district = side // n_districts
    for node in g.nodes:
        _row, col = divmod(node, side)
        g.nodes[node]["TOTPOP"] = 1
        g.nodes[node]["district"] = col // cols_per_district
    # Rebuild with nodes inserted in a random order, so the *stored* order has no
    # spatial locality (attributes and edges are preserved untouched).
    shuffled = list(g.nodes)
    random.Random(shuffle_seed).shuffle(shuffled)
    h = nx.Graph()
    h.add_nodes_from((node, g.nodes[node]) for node in shuffled)
    h.add_edges_from(g.edges)
    return h


grid = build_grid_graph(SIDE, N_DISTRICTS)
GRAPH_PATH.write_text(json.dumps(nx.readwrite.json_graph.adjacency_data(grid)))
print(
    f"graph file: {GRAPH_PATH} ({GRAPH_PATH.stat().st_size} bytes, {SIDE * SIDE} nodes)"
)

graph file: example_data/grid.json (95290 bytes, 1024 nodes)


### The public surface

Everything bundle-related is re-exported from the top-level package, but it
lives in two submodules:

- `binary_ensemble.bundle` — `BendlEncoder`, `BendlDecoder`, `compress_stream`
- `binary_ensemble.graph` — `reorder`, `reorder_multi_level_cluster`,
  `reorder_reverse_cuthill_mckee`, `reorder_by_key`

(The plain-stream `BenEncoder` / `BenDecoder` and the whole-file `encode_*` /
`decode_*` codec helpers are the subject of the BEN tutorial.)

In [2]:
from binary_ensemble import BendlDecoder, BendlEncoder, compress_stream
from binary_ensemble import graph as bgraph

## The GerryChain ingredients

We drive everything with a short ReCom chain. The chain's *recipe* (proposal,
constraints, updaters) is independent of how nodes are ordered, so we factor it
into a helper that builds a fresh chain on whatever graph we hand it. We'll call
this once per bundle and **stream each plan to disk as the chain produces it** —
no need to hold the whole ensemble in memory.

In [3]:
from functools import partial

from gerrychain import Graph, MarkovChain, Partition, accept, constraints, updaters
from gerrychain.proposals import recom


def make_chain(gc_graph, steps):
    """Build a fresh ReCom MarkovChain over ``gc_graph`` (a gerrychain.Graph)."""
    chain_updaters = {
        "population": updaters.Tally("TOTPOP", alias="population"),
        "cut_edges": updaters.cut_edges,
    }
    initial = Partition(gc_graph, assignment="district", updaters=chain_updaters)
    ideal_pop = sum(initial["population"].values()) / len(initial)
    return MarkovChain(
        proposal=partial(
            recom, pop_col="TOTPOP", pop_target=ideal_pop, epsilon=0.05, node_repeats=2
        ),
        constraints=[constraints.contiguous],
        accept=accept.always_accept,
        initial_state=initial,
        total_steps=steps,
    )

## Writing your first bundle — encoding as the chain runs

You do **not** need to use `BendlEncoder` itself as a context manager. Only the
`stream(...)` writer needs a `with` block: closing the stream context is what
finalizes the bundle. So the pattern is:

1. create the encoder and add the graph (and any other assets),
2. open the single-use `stream(...)` in a `with` block,
3. iterate the chain and `write` each plan inside it,
4. when the `with enc.stream(...)` block exits, the bundle is finalized on disk.

The one rule when writing: every assignment must be in a **fixed, known node
order**. GerryChain makes no ordering promise, so we pin the order to the graph's
node iteration order and reindex each plan to it.

A convenient trick: `add_graph` *returns* the embedded graph (as a NetworkX
graph), so we can build the GerryChain graph straight from it and guarantee the
write order matches what gets stored. For this first bundle we pass
`sort=None` to store the graph in its raw (shuffled) order — a
deliberately un-optimized baseline we'll improve on next.

In [4]:
encoder = BendlEncoder("example_data/basic.bendl", overwrite=True)  # no `with` needed
stored_graph = encoder.add_graph(GRAPH_PATH, sort=None)
gc_graph = Graph.from_networkx(stored_graph)
write_order = list(gc_graph.nodes)  # the order stored == the order we write

with encoder.stream("ben") as stream:  # only the stream is context-managed
    for partition in make_chain(gc_graph, steps=1000):
        series = partition.assignment.to_series()
        stream.write(series.loc[write_order].astype(int).tolist())
# the bundle is finalized now that the stream context has closed

print("wrote example_data/basic.bendl")

wrote example_data/basic.bendl


A note on validation: because we embedded a graph *before* the stream, the
encoder knows the node count and checks every `write` against it. A
wrong-length assignment raises immediately instead of silently corrupting the
file (and because the exception escapes the stream context, the bundle is left
unfinalized rather than stamped complete — more on that at the end):

In [5]:
encoder = BendlEncoder("example_data/willfail.bendl", overwrite=True)
encoder.add_graph(GRAPH_PATH, sort=None)
try:
    with encoder.stream("ben") as stream:
        stream.write([0, 1, 2])  # too short
except ValueError as e:
    print("rejected as expected:", e)

rejected as expected: assignment length 3 does not match graph node count 1024


## Reordering for compression (the default)

BEN/XBEN compress *runs of equal adjacent labels* well, so a node ordering that
keeps neighbouring nodes near each other in the stream compresses much better.
Because our grid's stored order is shuffled, the raw `basic.bendl` above is a
worst case. Fixing it is the encoder's default behaviour: `add_graph` reorders
the graph with **multi-level clustering (`sort="mlc"`)** unless you opt out with
`sort=None`. Reordering:

- reorders the graph — `sort="mlc"` (default), `sort="rcm"`, or `sort="key"`
  with `key="<attribute>"` (e.g. `key="GEOID"`) to sort by a node attribute,
- stores both the reordered `graph.json` **and** a `node_permutation_map.json`,
- and **returns the reordered graph**.

Returning the reordered graph is what makes this ergonomic: we build the *entire
ReCom chain on that ordering*, so the chain's natural node order already equals
the stored order — streaming needs no extra bookkeeping. **Reordering is
pre-stream only** (it decides the write order), so `add_graph(...)` must come
before `stream()`.

We'll make this the "real" bundle for the rest of the tutorial, so we also stamp
in metadata and a couple of custom assets while we're here.

In [6]:
encoder = BendlEncoder("example_data/rich.bendl", overwrite=True)

# add_graph reorders with MLC by default; build the chain on the returned graph.
reordered_graph = encoder.add_graph(GRAPH_PATH)
gc_graph = Graph.from_networkx(reordered_graph)
write_order = list(gc_graph.nodes)

# Provenance + extra assets (covered in detail in the next section).
encoder.add_metadata(
    {"generator": "gerrychain", "proposal": "recom", "epsilon": 0.05, "seed": 1234}
)
encoder.add_asset(
    "readme.txt", "ReCom ensemble on a 32x32 grid, MLC-reordered.", "text"
)

with encoder.stream("ben") as stream:
    for partition in make_chain(gc_graph, steps=1000):
        series = partition.assignment.to_series()
        stream.write(series.loc[write_order].astype(int).tolist())

print("wrote example_data/rich.bendl")

wrote example_data/rich.bendl


### Did reordering actually help?

Tempting as it is to compare `basic.bendl` against `rich.bendl`, that isn't a
fair fight: they hold **different ensembles** — each was streamed live from its
own independent ReCom run — so their stream sizes mix the ordering effect with
run-to-run randomness. Let's look anyway, then do it properly. We compare the
*embedded BEN stream* sizes (the assignment data only, excluding assets and
header) by extracting each stream and measuring it:

In [7]:
import os


def stream_size(path):
    """Size in bytes of a bundle's embedded BEN stream (extracted)."""
    decoder = BendlDecoder(path)
    tmp = "example_data/_measure.ben"
    decoder.extract_stream(tmp, overwrite=True)
    size = os.path.getsize(tmp)
    os.remove(tmp)
    return size


print(f"basic.bendl (raw,  run A): {stream_size('example_data/basic.bendl'):>8} bytes")
print(f"rich.bendl  (mlc,  run B): {stream_size('example_data/rich.bendl'):>8} bytes")

basic.bendl (raw,  run A):   135840 bytes
rich.bendl  (mlc,  run B):    40081 bytes


For a true **apples-to-apples** measurement we need the *same* plans in two
orderings. We can get that without running a second chain by **relabeling**
`basic.bendl`'s exact ensemble into MLC order. `relabel_bundle` does exactly
this in one call: it reorders the stored graph, rewrites every assignment into
the new node order, and stores a `node_permutation_map.json` so the change stays
reversible (it preserves metadata and custom assets too). It's the bundle-level
form of the CLI's `reben` ordering step:

In [8]:
from binary_ensemble import relabel_bundle

# out_file won't overwrite an existing file, so clear any copy from a previous run.
Path("example_data/relabeled.bendl").unlink(missing_ok=True)
relabel_bundle(
    "example_data/basic.bendl", out_file="example_data/relabeled.bendl", sort="mlc"
)

raw_bytes = stream_size("example_data/basic.bendl")
mlc_bytes = stream_size("example_data/relabeled.bendl")
print(f"same ensemble, raw order: {raw_bytes:>8} bytes")
print(f"same ensemble, MLC order: {mlc_bytes:>8} bytes")
print(f"-> {raw_bytes / mlc_bytes:.1f}x smaller from reordering alone")

same ensemble, raw order:   135840 bytes
same ensemble, MLC order:    39908 bytes
-> 3.4x smaller from reordering alone


Now the *only* thing that changed is the node ordering, so that ratio is the
real compression win from MLC — and it's why MLC is the **default** in
`add_graph`. (On a graph that already arrives in a locality-friendly order the
gain is smaller, and the extra `node_permutation_map.json` can even make a tiny
file net-larger, but reordering is cheap and rarely hurts — so the encoder does
it for you unless you ask for raw with `sort=None`.) It matters most
right before an expensive XBEN recompress, where every byte of BEN is amplified.

### Reordering under the hood: the standalone utilities

`add_graph(..., sort=..., key=...)` is built on the `binary_ensemble.graph`
utilities, which you can also call directly — handy when you want to compute an
ordering once and reuse it, or inspect the permutation map before committing.
Each returns `(reordered_graph, node_permutation_map)`: a live NetworkX graph
plus the map dict.

In [9]:
reordered, permutation_map = bgraph.reorder(GRAPH_PATH, sort="rcm")
print(
    "reorder(sort='rcm') ->",
    type(reordered).__name__,
    "with",
    reordered.number_of_nodes(),
    "nodes",
)

# Sort by a node attribute with sort="key" + key=...  (on real data this is how
# you'd order by, say, "GEOID"; here the grid only has "district"/"id"):
graph_mlc, _ = bgraph.reorder(GRAPH_PATH, sort="mlc")
graph_rcm, _ = bgraph.reorder(GRAPH_PATH, sort="rcm")
graph_by_district, _ = bgraph.reorder(GRAPH_PATH, sort="key", key="district")
# reorder_multi_level_cluster / reorder_reverse_cuthill_mckee / reorder_by_key are
# thin convenience wrappers over these.
print("orderings: sort='mlc', sort='rcm', or sort='key' with key='<attribute>'")

# The permutation map is what makes a reordering reversible: its required field
# `node_permutation_old_to_new` maps original 0-based node positions -> new ones.
old_to_new = permutation_map["node_permutation_old_to_new"]
print(
    "old_to_new is a bijection over [0, n):",
    sorted(old_to_new.values()) == list(range(reordered.number_of_nodes())),
)
print("provenance fields:", {k: permutation_map[k] for k in ("ordering_method", "key")})

reorder(sort='rcm') -> Graph with 1024 nodes
orderings: sort='mlc', sort='rcm', or sort='key' with key='<attribute>'
old_to_new is a bijection over [0, n): True
provenance fields: {'ordering_method': 'reverse-cuthill-mckee', 'key': None}


## Metadata and custom assets

We already used these while building `rich.bendl`. `add_metadata` writes the
canonical `metadata.json` (provenance). `add_asset` writes a *custom* asset
under a name you choose, with a `content_type` of `"json"` or `"text"`:

- `"json"` — payload must be valid UTF-8 JSON; the decoder will auto-parse it.
- `"text"` — payload must be valid UTF-8; stored without the JSON flag.

The facade validates the payload, so a malformed `"json"` asset is caught at
write time. Assets may be added before *or* after the stream — only the stream
itself is single-use. Post-stream adds commit immediately (one directory
rewrite each), so use them sparingly. Here we tack a JSON asset onto an
already-finalized bundle to show both behaviours:

In [10]:
# Add to rich.bendl after the fact (this finalized bundle is reopened to append).
# In append mode each add_* commits immediately, so there is nothing to finalize.
appender = BendlEncoder.append("example_data/rich.bendl")
appender.add_asset("params.json", json.dumps({"node_repeats": 2}), "json")

# Validation in action — a "json" asset that isn't JSON is rejected up front:
encoder = BendlEncoder("example_data/tmp.bendl", overwrite=True)
try:
    encoder.add_asset("bad.json", "this is not json", "json")
except ValueError as e:
    print("rejected as expected:", e)

rejected as expected: content_type='json' requires valid UTF-8 JSON: Expecting value: line 1 column 1 (char 0)


## Reading a bundle

`BendlDecoder(path)` opens a bundle. The **canonical getters** pull the
well-known assets back in convenient form:

- `read_graph()` → a live **NetworkX graph** (or `None` if absent),
- `read_metadata()` → parsed `metadata.json` (or `None`),
- `read_node_permutation_map()` → parsed map dict (or `None`).

Crucially, `read_graph()` returns the graph in the node order the assignments
were written in — which, because we built the chain on the reordered graph, is
exactly the reordered order. It lines up with the stream with no extra work.

In [11]:
decoder = BendlDecoder("example_data/rich.bendl")

packaged_graph = decoder.read_graph()
print(
    "read_graph() ->",
    type(packaged_graph).__name__,
    "with",
    packaged_graph.number_of_nodes(),
    "nodes",
)
print("read_metadata() ->", decoder.read_metadata())
print(
    "read_node_permutation_map() has old_to_new:",
    "node_permutation_old_to_new" in decoder.read_node_permutation_map(),
)

read_graph() -> Graph with 1024 nodes
read_metadata() -> {'generator': 'gerrychain', 'proposal': 'recom', 'epsilon': 0.05, 'seed': 1234}
read_node_permutation_map() has old_to_new: True


**Generic accessors** reach any asset by name:

- `read_asset_bytes(name)` → raw `bytes`,
- `read_json_asset(name)` → parsed JSON.

Note `read_json_asset("graph.json")` gives you the *raw* adjacency dict, in case
you want the JSON rather than the rebuilt NetworkX object.

In [12]:
print("readme.txt   ->", decoder.read_asset_bytes("readme.txt"))
print("params.json  ->", decoder.read_json_asset("params.json"))
print(
    "graph.json (raw dict) top-level keys:",
    list(decoder.read_json_asset("graph.json").keys()),
)

readme.txt   -> b'ReCom ensemble on a 32x32 grid, MLC-reordered.'
params.json  -> {'node_repeats': 2}
graph.json (raw dict) top-level keys: ['directed', 'multigraph', 'graph', 'nodes', 'adjacency']


## Inspecting a bundle

Before (or instead of) reading payloads, you can inspect structure — handy for
tooling, debugging, or deciding whether a file is what you expect:

- `version` → `(major, minor)` format version,
- `is_complete()` → was it finalized cleanly,
- `assignment_format()` → `"ben"` or `"xben"`,
- `asset_names()` → directory names in order,
- `list_assets()` → full directory: name, type, offset, len, flag tags,
- `len(dec)` / `count_samples()` → number of plans in the stream.

In [13]:
decoder = BendlDecoder("example_data/rich.bendl")
print("version:          ", decoder.version())
print("is_complete:      ", decoder.is_complete())
print("assignment_format:", decoder.assignment_format())
print("sample count:     ", len(decoder))
print("asset_names:      ", decoder.asset_names())
print("full directory:")
for entry in decoder.list_assets():
    print("   ", entry)

version:           (1, 0)
is_complete:       True
assignment_format: ben
sample count:      1000
asset_names:       ['graph.json', 'node_permutation_map.json', 'metadata.json', 'readme.txt', 'params.json']
full directory:
    {'name': 'graph.json', 'type': 2, 'offset': 64, 'len': 6788, 'flags': ['json', 'xz', 'checksum']}
    {'name': 'node_permutation_map.json', 'type': 3, 'offset': 6852, 'len': 10152, 'flags': ['json', 'checksum']}
    {'name': 'metadata.json', 'type': 1, 'offset': 17004, 'len': 79, 'flags': ['json', 'checksum']}
    {'name': 'readme.txt', 'type': 4, 'offset': 17083, 'len': 46, 'flags': ['checksum']}
    {'name': 'params.json', 'type': 4, 'offset': 57400, 'len': 19, 'flags': ['json', 'checksum']}


## Iterating the stream and reconstructing plans

A `BendlDecoder` iterates its embedded stream, yielding each assignment as a
`list[int]`. Combined with `read_graph()`, you can rebuild GerryChain
`Partition`s straight from the bundle — no separate graph file, no remembered
node order:

In [14]:
import pandas as pd

decoder = BendlDecoder("example_data/rich.bendl")
packaged_graph = decoder.read_graph()
order = pd.Index(packaged_graph.nodes)  # matches the written assignment order

cut_edge_counts = []
for assignment in decoder:
    partition = Partition(
        packaged_graph,
        assignment=pd.Series(assignment, index=order),
        updaters={"cut_edges": updaters.cut_edges},
    )
    cut_edge_counts.append(len(partition["cut_edges"]))

print(f"reconstructed {len(cut_edge_counts)} partitions from the bundle alone")
print("first five cut-edge counts:", cut_edge_counts[:5])

reconstructed 1000 partitions from the bundle alone
first five cut-edge counts: [96, 87, 96, 100, 122]


## Subsampling

For winnowing a large ensemble you rarely want every plan. `BendlDecoder`
supports three native subsamplers; each returns the decoder set up to yield
only the chosen plans, so you still just iterate. **Indices are 1-based** (plan
1 is the first sample):

- `subsample_indices([...])` — exactly these 1-based indices (sorted, unique),
- `subsample_range(start, end)` — the 1-based *inclusive* range `[start, end]`,
- `subsample_every(step, offset=1)` — every `step`-th plan starting at `offset`
  ("thinning").

In [15]:
bundle_file = "example_data/rich.bendl"
decoder = BendlDecoder(bundle_file)  # one decoder, reused for every subsample below

print(
    "indices [1, 500, 1000] ->",
    [assignment[:4] for assignment in decoder.subsample_indices([1, 500, 1000])],
)
print(
    "range(100, 104)        ->",  # plans 100..104 inclusive = 5 plans
    [assignment[:4] for assignment in decoder.subsample_range(100, 104)],
)
print(
    "every 250th            ->", sum(1 for _ in decoder.subsample_every(250)), "plans"
)

# The same decoder rewinds and re-selects on each call, so you can run subsamples
# repeatedly without building a new decoder:
print(
    "indices again          ->",
    [assignment[:4] for assignment in decoder.subsample_indices([1, 500, 1000])],
)

indices [1, 500, 1000] -> [[2, 2, 2, 2], [3, 3, 3, 3], [2, 2, 2, 2]]
range(100, 104)        -> [[0, 0, 0, 0], [0, 0, 0, 0], [3, 3, 3, 3], [3, 3, 3, 3], [3, 3, 3, 3]]
every 250th            -> 4 plans
indices again          -> [[2, 2, 2, 2], [3, 3, 3, 3], [2, 2, 2, 2]]


## Extracting the raw stream

Sometimes you want the bare assignment stream back out — e.g. to hand it to the
plain-stream tools or a different pipeline. `extract_stream` copies the
embedded stream region verbatim to a standalone `.ben`/`.xben` file, which you
can then open with the stream-only `BenDecoder`.

In [16]:
from binary_ensemble import BenDecoder

decoder = BendlDecoder("example_data/rich.bendl")
decoder.extract_stream("example_data/extracted.ben", overwrite=True)

# Open the extracted file with the plain stream decoder (mode matches the bundle).
ben = BenDecoder("example_data/extracted.ben", mode=decoder.assignment_format())
print("extracted stream yields", sum(1 for _ in ben), "plans")

extracted stream yields 1000 plans


## Appending analysis back onto the bundle

A finished, finalized bundle isn't frozen: `BendlEncoder.append(path)` opens it
to add more assets later — say, the cut-edge summary we just computed. The
stream is *not* re-opened (it's already written); each `add_*` commits
immediately to disk.

In [17]:
appender = BendlEncoder.append("example_data/rich.bendl")
appender.add_asset(
    "cut_edge_summary.json",
    json.dumps(
        {
            "mean": sum(cut_edge_counts) / len(cut_edge_counts),
            "min": min(cut_edge_counts),
            "max": max(cut_edge_counts),
        }
    ),
    "json",
)

decoder = BendlDecoder("example_data/rich.bendl")
print("assets after append:", decoder.asset_names())
print("appended summary:", decoder.read_json_asset("cut_edge_summary.json"))

assets after append: ['graph.json', 'node_permutation_map.json', 'metadata.json', 'readme.txt', 'params.json', 'cut_edge_summary.json']
appended summary: {'mean': 130.707, 'min': 87, 'max': 186}


## Assets-only bundles (no stream)

You don't have to write a stream at all. This is the one case where you finalize
the bundle yourself — since there's no `stream()` context to do it — with an
explicit `close()` (or by using the encoder as a context manager). The result is
a valid **assets-only** bundle, useful for shipping a graph + metadata package
on its own. It decodes to an empty iteration with `len == 0` (no spurious
"missing stream" error).

In [18]:
encoder = BendlEncoder("example_data/assets_only.bendl", overwrite=True)
encoder.add_graph(GRAPH_PATH, sort=None)
encoder.add_metadata({"note": "graph package, no plans"})
encoder.close()  # no stream was opened, so finalize explicitly

decoder = BendlDecoder("example_data/assets_only.bendl")
print(
    "assets-only: is_complete =",
    decoder.is_complete(),
    "| len =",
    len(decoder),
    "| assets =",
    decoder.asset_names(),
)

assets-only: is_complete = True | len = 0 | assets = ['graph.json', 'metadata.json']


## Recompressing to XBEN for archival

BEN is fast to write and good for active work. For long-term storage, XBEN
squeezes much harder (at a real CPU/time cost). `compress_stream` repackages a
bundle's BEN stream as XBEN, **preserving every asset** (graph, metadata,
permutation map, custom blobs). Choose exactly one of:

- `in_place=True` — recompress to a temp file and atomically swap it in, or
- `out_file=...` — write a new bundle and leave the original untouched.

In [19]:
# Write a fresh XBEN copy, original preserved. (out_file won't overwrite an
# existing file, so clear any copy from a previous run first.)
Path("example_data/rich-archive.bendl").unlink(missing_ok=True)
compress_stream("example_data/rich.bendl", out_file="example_data/rich-archive.bendl")

xben_decoder = BendlDecoder("example_data/rich-archive.bendl")
print("recompressed format:", xben_decoder.assignment_format())
print("assets preserved:   ", xben_decoder.asset_names())
print("metadata preserved: ", xben_decoder.read_metadata())
print(
    "plans unchanged:    ",
    len(xben_decoder),
    "==",
    len(BendlDecoder("example_data/rich.bendl")),
)

recompressed format: xben
assets preserved:    ['graph.json', 'node_permutation_map.json', 'metadata.json', 'readme.txt', 'params.json', 'cut_edge_summary.json']
metadata preserved:  {'generator': 'gerrychain', 'proposal': 'recom', 'epsilon': 0.05, 'seed': 1234}
plans unchanged:     1000 == 1000


/tmp/claude-1000/ipykernel_3730522/3018985229.py:6: UserWarning: XBEN may take a second to start decoding.
  xben_decoder = BendlDecoder("example_data/rich-archive.bendl")


(Passing both `in_place=True` and `out_file=`, or neither, raises — the choice
is exclusive. Note XBEN bundles emit a one-time startup warning on decode,
since opening them does real decompression work.)

## Lifecycle and failure semantics

A subtle but important guarantee: if an exception escapes the `stream()`
context — say the chain or your write logic throws partway through — the bundle
is left **unfinalized** rather than stamped complete over a half-written
stream. You can detect this (`is_complete()` is `False`) and still recover what
was written via `extract_stream(..., allow_unfinalized=True)`.

In [20]:
encoder = BendlEncoder("example_data/partial.bendl", overwrite=True)
stored_graph = encoder.add_graph(GRAPH_PATH, sort=None)
gc_graph = Graph.from_networkx(stored_graph)
write_order = list(gc_graph.nodes)
try:
    with encoder.stream("ben") as stream:
        for i, partition in enumerate(make_chain(gc_graph, steps=1000)):
            if i == 50:
                raise RuntimeError("simulated crash mid-stream")
            series = partition.assignment.to_series()
            stream.write(series.loc[write_order].astype(int).tolist())
except RuntimeError as e:
    print("caught:", e)

decoder = BendlDecoder("example_data/partial.bendl")
print("is_complete:", decoder.is_complete(), "(left unfinalized, as intended)")
decoder.extract_stream(
    "example_data/partial.ben", overwrite=True, allow_unfinalized=True
)
recovered = sum(1 for _ in BenDecoder("example_data/partial.ben", mode="ben"))
print("recovered", recovered, "plans written before the crash")

caught: simulated crash mid-stream
is_complete: False (left unfinalized, as intended)
recovered 50 plans written before the crash


## Recap — when to reach for what

- **`BendlEncoder` / `BendlDecoder`** are your default for storing an ensemble:
  one self-describing file, graph + metadata included, encoded live as the
  chain runs. You only ever need a `with` block around the `stream()` writer —
  closing it finalizes the bundle (use `close()` for an assets-only bundle).
- **`add_graph(graph)`** before the stream (MLC-reordered by default; pass
  `sort="rcm"`, `sort="key", key="GEOID"`, or `sort=None` for raw), then build
  the chain on the returned graph — you get a compression win *and* a write order
  that already matches the stored graph.
- **`relabel_bundle`** to reorder an *existing* BEN bundle and rewrite its stream
  to match (in place or to a new file) — e.g. to optimize a bundle you received
  raw, before archiving it.
- **`binary_ensemble.graph.reorder*`** when you want the reordering standalone
  (e.g. to reuse an ordering across several bundles).
- **`add_metadata` / `add_asset`** to stamp provenance and ship analysis
  alongside the plans; **`append`** to add results to a finished bundle.
- **`compress_stream`** to graduate an active BEN bundle to an archival XBEN
  one without losing any asset.
- Drop to the plain **`binary_ensemble.stream`** API (via `extract_stream`)
  only when you specifically need the bare stream and are tracking the graph
  and node order yourself.
print("done — see the example_data/ folder for the bundles this tutorial wrote")